## Request Data from Coinalyse API
Documents: https://api.coinalyze.net/v1/doc/

### 1 - Import Libraries

In [1]:
import json as jn
import requests as rq
from datetime import datetime as dt
import pandas as pd

### 2 - Parametres pre-definition

In [ ]:
# the base url
url = "https://api.coinalyze.net/v1/"

# list of data to call from api
url_espec = ["ohlcv-history", "long-short-ratio-history", "liquidation-history", "predicted-funding-rate-history",  "funding-rate-history", "open-interest-history"]

# the key to access the api
api_key = {"api_key" : "secret"}

# this correspond to the first moment of the bitcoin in the exchange in miliseconds
from_f = 1568260800

# now format in date time
to_t = int(dt.now().timestamp())

btc = 'BTCUSDT'

### 3 - Calling from API -- Bitcoin from Binance (BTC/USDT)

#### 3.1 - Exchanges

In [3]:
# call the api for the name and code for each exchange
exchanges = rq.get(url + 'exchanges', headers=api_key)
# saves in dataframe
exchanges_df = pd.DataFrame(data=exchanges.json())

In [4]:
exchanges_df.head()

,name,code
0,Poloniex,P
1,Vertex,V
2,Bitforex,D
3,Kraken,K
4,Bithumb,U


In [5]:
# retorn the code for binance ('A')
binance_code = exchanges_df[exchanges_df['name'] == 'Binance']['code'].iloc[0]

#### 3.2 - Markets

In [6]:
# call the api for the trade pairs
market = rq.get(url + 'future-markets', headers=api_key)
# saves in dataframe
market_df = pd.DataFrame(data=market.json())

In [7]:
market_df.head()

,symbol,exchange,symbol_on_exchange,base_asset,quote_asset,expire_at,has_buy_sell_data,is_perpetual,margined,oi_lq_vol_denominated_in,has_long_short_ratio_data,has_ohlcv_data
0,MNTUSDT.6,6,MNTUSDT,MNT,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
1,QNTUSDT.6,6,QNTUSDT,QNT,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
2,MAVUSDT.6,6,MAVUSDT,MAV,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
3,NMRUSDT_PERP.A,A,NMRUSDT,NMR,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
4,L3USDT.6,6,L3USDT,L3,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True


In [8]:
# retorn the symbol for Bitcoin ('BTCUSDT_PERP.A')
btc_binance = market_df[(market_df['exchange'] == binance_code) & (market_df['symbol_on_exchange'] == btc)]['symbol'].iloc[0]

#### 3.3 - The Data

In [9]:
# the parametres that defines:
ohlc_params = {
            "symbols" : btc_binance, # the name of the pair (BTC/USD) futures from Binance
            "interval" : "daily", # the timeframe
            "from" : from_f, 
            "to" : to_t
            }

In [10]:
# the function to call the api
def data_dumper(endpoint):   
    response = rq.get(
                    f"{url}{endpoint}", 
                    params = ohlc_params, 
                    headers = api_key
                    )
    return response

In [11]:
url_espec[0]

'ohlcv-history'

In [12]:
# the dictionary where the data will be stored
data = {}

# a loop to call each individual data and put on the dictionary
for endpoint in url_espec:
    data[endpoint] = data_dumper(endpoint).json()

### 4 - Transform to a CSV file

In [13]:
# the function to put every call to a single dataframe
def df_maker(key, value, df_data):
    df = pd.DataFrame(data=value[0]['history'])
    
    # a loop to rename every column but 't'
    for c in df.columns:
        if c != 't':
            df.rename(columns={c: c + '_' + key}, inplace=True)
        
    # return the dataframe if is the first time    
    if df_data is None:
        return df
    # merge if is not
    else:
        return pd.merge(df_data, df, how='left', on='t')      

In [14]:
# initiate the dataframe
df_data = None

# a loop to transform every data to a single dataframe
for key, value in data.items():
    df_data = df_maker(key, value, df_data) 

### 5 - Check the result and save as CSV

In [17]:
df_data.tail()

,t,o_ohlcv-history,h_ohlcv-history,l_ohlcv-history,c_ohlcv-history,v_ohlcv-history,bv_ohlcv-history,tx_ohlcv-history,btx_ohlcv-history,r_long-short-ratio-history,...,l_predicted-funding-rate-history,c_predicted-funding-rate-history,o_funding-rate-history,h_funding-rate-history,l_funding-rate-history,c_funding-rate-history,o_open-interest-history,h_open-interest-history,l_open-interest-history,c_open-interest-history
2205,1758844800,108934.6,110250.0,108566.0,109588.5,111433.664,55716.173,1111514,551506.0,1.8035,...,-0.000117,-0.000028,0.003143,0.007716,0.003143,0.007716,84821.193,86660.664,84458.723,84919.841
2206,1758931200,109588.5,109700.0,109021.9,109577.3,35808.071,17875.676,342032,168645.0,1.7917,...,-0.000577,0.003332,-0.000010,0.007821,-0.000010,0.007821,84919.760,85127.114,84517.446,84550.888
2207,1759017600,109577.2,112300.0,109136.5,112119.6,77220.071,39810.148,614121,314616.0,1.7824,...,0.001616,0.002264,0.003314,0.005261,0.003314,0.005261,84550.926,86224.886,84418.636,86090.117
2208,1759104000,112119.7,114377.2,111501.0,114257.1,125365.977,63032.434,983709,487959.0,1.4900,...,-0.001262,0.002353,0.002312,0.002312,0.000032,0.000032,86089.696,88987.440,85641.969,88865.851
2209,1759190400,114257.1,114800.0,112615.3,113458.0,76146.515,38362.285,668295,335384.0,1.0227,...,0.001904,0.004686,0.002353,0.005893,0.002353,0.005893,88864.528,90374.025,88243.557,88247.935


In [16]:
# saves in CSV file    
df_data.to_csv("futures_raw_data.csv", index=False)  